# SignVisionAI - Member 2: Machine Learning Model Development
### A–Z Sign Language Recognition using SLS Hand Landmark Coordinates

**Author:** Member 2 (ML Model Development)  
**Objective:** Build, tune, evaluate, and compare Machine Learning models (**Support Vector Machine**, **Random Forest**, and **Multi-Layer Perceptron**) to classify Sri Lankan Sign Language (SLS) / ASL A–Z hand signs using 21 MediaPipe hand landmark coordinates.

---
### Complete ML Workflow:
1. **Load and Prepare Dataset** (`sls_hand_landmarks_clean.csv`)
2. **Stratified Train / Test Split** (80% Train, 20% Held-out Test, `random_state=42`)
3. **Model Pipelines with Preprocessing** (`StandardScaler` + Classifiers to prevent data leakage)
4. **5-Fold Stratified Cross-Validation & Hyperparameter Tuning** (`GridSearchCV` on training set only)
5. **Held-out Test Evaluation** (Accuracy, Precision, Recall, F1-Score, Classification Reports)
6. **Model Comparison & Confusion Matrix Visualizations**
7. **Best Model Selection & Serialization** (`models/sls_best_model.joblib`)

## 0. Import Required Libraries and Setup Paths

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Constants for Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

# Create output directories
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

print("Libraries loaded and directories ready.")

## 1. Load and Prepare the Dataset
- Target file: `sls_hand_landmarks_clean.csv`
- Features: 63 landmark coordinates (`lm0_x, lm0_y, lm0_z, ... , lm20_x, lm20_y, lm20_z`)
- Target: `label` (A–Z classes)

In [ ]:
# Dataset path resolution
candidate_paths = [
    "../data/sls_hand_landmarks_clean.csv",
    "data/sls_hand_landmarks_clean.csv",
    "../data/asl_hand_landmarks (1).csv",
    "../data/asl_hand_landmarks.csv",
    "../data/sls_hand_landmarks.csv"
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# Identify label column
if "label" in df.columns:
    target_col = "label"
elif "Label" in df.columns:
    target_col = "Label"
else:
    target_col = df.columns[-1] if df.dtypes.iloc[-1] == object else df.columns[0]

# Clean label strings and filter to valid A-Z alphabet
df[target_col] = df[target_col].astype(str).str.strip().str.upper()
valid_mask = df[target_col].str.match(r"^[A-Z]$")
df = df[valid_mask].copy()

X = df.drop(columns=[target_col]).astype(np.float32)
y = df[target_col]

print("=" * 60)
print(f"Dataset Rows (Samples): {X.shape[0]}")
print(f"Dataset Columns (Features): {X.shape[1]} (21 coordinates x 3 [x, y, z])")
print(f"Number of Unique Classes : {y.nunique()}")
print(f"Missing Feature Values   : {X.isnull().sum().sum()}")
print(f"Missing Target Values    : {y.isnull().sum()}")
print("=" * 60)

df.head()

## 2. Train / Test Split (Stratified)
We split the dataset into an 80% Training Set and a 20% Held-out Test Set.
Stratification ensures every class from A to Z is equally represented in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training set samples : {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"Testing set samples  : {len(X_test)} ({len(X_test)/len(X):.1%})")
print(f"Classes represented  : {y_train.nunique()} in train, {y_test.nunique()} in test")

## 3. Define Model Pipelines & Search Spaces
We define scikit-learn `Pipeline` objects for:
1. **Support Vector Machine (SVM)** (`StandardScaler` + `SVC`)
2. **Random Forest (RF)** (`RandomForestClassifier`)
3. **Multi-Layer Perceptron (MLP)** (`StandardScaler` + `MLPClassifier`)

> **Data Leakage Prevention:** Feature scaling (`StandardScaler`) is part of the Pipeline, ensuring parameters are learned only from the training folds during cross-validation.

In [ ]:
models_config = {
    "Support Vector Machine (SVM)": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(random_state=RANDOM_STATE, probability=True))
        ]),
        "param_grid": {
            "classifier__C": [0.1, 1.0, 10.0, 50.0],
            "classifier__gamma": ["scale", "auto", 0.01, 0.1],
            "classifier__kernel": ["rbf", "linear"]
        }
    },
    "Random Forest": {
        "pipeline": Pipeline([
            ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        "param_grid": {
            "classifier__n_estimators": [100, 200, 300],
            "classifier__max_depth": [None, 15, 25],
            "classifier__min_samples_split": [2, 5]
        }
    },
    "Multi-Layer Perceptron (MLP)": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE))
        ]),
        "param_grid": {
            "classifier__hidden_layer_sizes": [(128, 64), (128, 64, 32), (64, 32)],
            "classifier__activation": ["relu", "tanh"],
            "classifier__alpha": [0.0001, 0.001, 0.01]
        }
    }
}

print("Model pipelines and parameter search spaces initialized.")

## 4. Hyperparameter Tuning with 5-Fold Stratified Cross-Validation
We tune and train each model on the **training set only** using `GridSearchCV` and 5-Fold Stratified CV.

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
trained_models = {}
comparison_records = []
test_predictions = {}

for name, config in models_config.items():
    print(f"\n>>> Tuning & Cross-Validating: {name}...")
    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["param_grid"],
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train, y_train)
    
    best_estimator = grid_search.best_estimator_
    best_cv_score = grid_search.best_score_
    best_params = grid_search.best_params_
    
    print(f"   Best 5-Fold CV Macro F1: {best_cv_score:.4f}")
    print(f"   Optimal Hyperparameters: {best_params}")
    
    # Evaluate on held-out test set
    y_pred = best_estimator.predict(X_test)
    test_predictions[name] = y_pred
    
    acc = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    
    trained_models[name] = {
        "model": best_estimator,
        "best_params": best_params,
        "cv_score": best_cv_score
    }
    
    comparison_records.append({
        "Model": name,
        "5-Fold CV Macro F1": best_cv_score,
        "Test Accuracy": acc,
        "Test Macro Precision": prec_macro,
        "Test Macro Recall": rec_macro,
        "Test Macro F1": f1_macro,
        "Test Weighted F1": f1_weighted
    })

## 5. Model Performance Comparison Table

In [ ]:
comparison_df = pd.DataFrame(comparison_records)
formatted_table = comparison_df.copy()
formatted_table["Test Accuracy"] = formatted_table["Test Accuracy"].apply(lambda x: f"{x*100:.2f}%")
formatted_table["5-Fold CV Macro F1"] = formatted_table["5-Fold CV Macro F1"].apply(lambda x: f"{x:.4f}")
formatted_table["Test Macro Precision"] = formatted_table["Test Macro Precision"].apply(lambda x: f"{x:.4f}")
formatted_table["Test Macro Recall"] = formatted_table["Test Macro Recall"].apply(lambda x: f"{x:.4f}")
formatted_table["Test Macro F1"] = formatted_table["Test Macro F1"].apply(lambda x: f"{x:.4f}")
formatted_table["Test Weighted F1"] = formatted_table["Test Weighted F1"].apply(lambda x: f"{x:.4f}")

print("=" * 85)
print("                      FINAL MODEL PERFORMANCE COMPARISON TABLE                       ")
print("=" * 85)
print(formatted_table.to_string(index=False))
print("=" * 85)

# Save comparison table to results directory
comparison_df.to_csv("../results/sls_model_comparison.csv", index=False)
print("Saved comparison CSV to: results/sls_model_comparison.csv")
formatted_table

## 6. Detailed Classification Reports & Confusion Matrices

In [ ]:
# Print per-class precision, recall, f1-score
for name, y_pred in test_predictions.items():
    print(f"\n================================================================")
    print(f" Classification Report: {name}")
    print(f"================================================================")
    print(classification_report(y_test, y_pred, zero_division=0))

# Side-by-side Confusion Matrix comparison
classes = sorted(y_test.unique())
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle("Sign Language SLS Hand Landmark Recognition - Model Comparison", fontsize=16, fontweight="bold")

for ax, (name, y_pred) in zip(axes, test_predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation="vertical")
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    ax.set_title(f"{name}\nAcc: {acc*100:.2f}% | Macro F1: {f1:.4f}", fontsize=12)
    ax.set_xlabel("Predicted Sign Class", fontsize=10)
    ax.set_ylabel("True Sign Class", fontsize=10)

plt.tight_layout()
plt.savefig("../results/sls_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved combined confusion matrices to: results/sls_confusion_matrices.png")

## 7. Best Model Selection & Serialization (`Joblib`)
We select the best performing model (highest CV & Test Macro F1) and save the entire fitted pipeline.

In [ ]:
# Select best model
best_row = comparison_df.sort_values(by=["Test Macro F1", "Test Accuracy"], ascending=False).iloc[0]
best_name = best_row["Model"]
best_pipeline = trained_models[best_name]["model"]

print(f" Winning Model Selected : {best_name}")
print(f"   Test Accuracy        : {best_row['Test Accuracy']*100:.2f}%")
print(f"   Test Macro F1-Score  : {best_row['Test Macro F1']:.4f}")

# Plot Best Model Confusion Matrix
classes = sorted(y_test.unique())
y_pred_best = best_pipeline.predict(X_test)
cm_best = confusion_matrix(y_test, y_pred_best, labels=classes)

fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=classes)
disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation="vertical")
ax.set_title(f"Best Model Confusion Matrix: {best_name}\nAccuracy: {best_row['Test Accuracy']*100:.2f}% | Macro F1: {best_row['Test Macro F1']:.4f}", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Sign Class", fontsize=12)
ax.set_ylabel("True Sign Class", fontsize=12)
plt.tight_layout()
plt.savefig("../results/sls_best_model_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

# Save pipeline to models/
model_save_path = "../models/sls_best_model.joblib"
joblib.dump(best_pipeline, model_save_path)
print(f"\nSaved complete fitted pipeline to: {model_save_path}")

# Save metadata JSON
metadata = {
    "model_name": best_name,
    "classes": [str(c) for c in classes],
    "num_classes": len(classes),
    "feature_count": int(X_train.shape[1]),
    "best_hyperparameters": str(trained_models[best_name]["best_params"]),
    "test_metrics": {
        "accuracy": float(best_row["Test Accuracy"]),
        "macro_precision": float(best_row["Test Macro Precision"]),
        "macro_recall": float(best_row["Test Macro Recall"]),
        "macro_f1": float(best_row["Test Macro F1"]),
        "weighted_f1": float(best_row["Test Weighted F1"])
    },
    "pipeline_steps": [name for name, _ in best_pipeline.steps]
}
with open("../models/sls_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
print("Saved metadata to: models/sls_model_metadata.json")

## 8. Model Verification & Inference Test

In [ ]:
# Reload model from disk and test prediction
reloaded_pipeline = joblib.load(model_save_path)

sample_x = X_test.iloc[0:1]
true_label = y_test.iloc[0]
pred_label = reloaded_pipeline.predict(sample_x)[0]

print("Verification on Held-out Sample:")
print(f" -> Ground Truth Sign : {true_label}")
print(f" -> Model Prediction   : {pred_label}")
print(f" -> Status             : {'MATCH - SUCCESS' if true_label == pred_label else 'MISMATCH'}")